CUSTOMER CHURN DETECTION PROJECT

In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import joblib

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import BernoulliNB

In [25]:
from sklearn.ensemble import VotingClassifier, StackingClassifier

In [14]:
import os
print(os.getcwd())
print(os.listdir())

C:\Users\Gaurav.mathur\Customer Churn Prediction\notebooks
['.ipynb_checkpoints', 'Customer Churn Detection.ipynb']


In [15]:
#Import the dataset
df = pd.read_csv("../data/telco_dataset.csv")
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [29]:
#Set Features(X) and Target(y)
X = df.drop(columns = 'Churn')
y = df['Churn'].map({
    'No': 0,
    'Yes': 1
})

In [30]:
#train test split
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size = 0.2,
                                                    random_state = 42,
                                                    stratify = y
                                                   )

In [31]:
#Separate numerical and categorical columns and then create preprocessor for encoding
categorical_cols = X.select_dtypes(include = 'object').columns.tolist()
numerical_cols = X.select_dtypes(exclude = 'object').columns.tolist()

preprocessor = ColumnTransformer(
    transformers = [
        (
            "cat",
    OneHotEncoder(handle_unknown = 'ignore'),
            categorical_cols
        ),
        (
            "num",
            StandardScaler(),
            numerical_cols
        )
    ]
)
print(preprocessor)

ColumnTransformer(transformers=[('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['gender', 'Partner', 'Dependents',
                                  'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
                                  'TechSupport', 'StreamingTV',
                                  'StreamingMovies', 'Contract',
                                  'PaperlessBilling', 'PaymentMethod']),
                                ('num', StandardScaler(),
                                 ['SeniorCitizen', 'tenure', 'MonthlyCharges',
                                  'TotalCharges'])])


In [32]:
#Create models [all models are created here to get the comparison among all and choose the best for testing]
#Logistic Regression
lr = LogisticRegression()

#Decision Tree
dt = DecisionTreeClassifier(random_state = 42)

#RandomForest
rf = RandomForestClassifier(random_state = 42)

#Gradient Boosting
gb = GradientBoostingClassifier(random_state = 42)

#XGBoost
xgb = XGBClassifier(random_state = 42)

#K Nearest Neighbors
knn = KNeighborsClassifier()

#SVM
svm = SVC(random_state = 42)

#Naive bayes
nb = BernoulliNB()

In [33]:
#Create a generic function for all pipelines
def create_pipeline(model, use_smote = True,
                    use_feature_selection = True,
                    K = 15):
    steps = []
    steps.append(('preprocessor', preprocessor))
    
    if use_smote:
        steps.append(('smote', SMOTE(random_state = 42)))

    if use_feature_selection:
        steps.append(('feature_selection', SelectKBest(score_func = mutual_info_classif, k = K)))

    steps.append(('model',model))
    return Pipeline(steps)

#Create all pipelines
lr_pipeline = create_pipeline(lr) #Pipeline for Logistic Regression

dt_pipeline = create_pipeline(dt) #Pipeline for decision tree

rf_pipeline = create_pipeline(rf) #Pipeline for random forest

gb_pipeline = create_pipeline(gb) #pipeline for gradient boosting

xgb_pipeline = create_pipeline(xgb) #pipeline for xgboost

knn_pipeline = create_pipeline(knn) #pipeline for kNeighbors

svm_pipeline = create_pipeline(svm) #pipeline for Support Vector Machine

nb_pipeline = create_pipeline(nb) #Pipeline for naive bayes (Bernoulli)

In [34]:
from sklearn.model_selection import GridSearchCV

In [35]:
#Generic function for tuning the pipelines
def tune_model(model, Param_grid):
    Grid_search = GridSearchCV(
        estimator = model,
        param_grid = Param_grid,
        scoring = 'f1',
        cv = 5,
        verbose = 2,
        n_jobs = -1
    )
    Grid_search.fit(X_train, y_train)

    final_model = Grid_search.best_estimator_
    print(f'Best parameters for {model}: \n', Grid_search.best_params_,'\n')
    return final_model

In [37]:
#Pipeline tuning(lr and nb don't require tuning)
dt_grid = {
    'model__max_depth': [3,4,5,6],
    'model__min_samples_split': [5, 10, 15, 20],
    'model__min_samples_leaf': [5,10,15,20]
}
dt_pipeline = tune_model(dt_pipeline, dt_grid)

rf_grid = {
    'model__max_depth' : [3,4,5,6],
    'model__n_estimators': [50,100,150,200],
    'model__min_samples_split': [5,10,15,20],
    'model__min_samples_leaf': [5,10,15,20],
    'model__max_features': ['sqrt']
}
rf_pipeline = tune_model(rf_pipeline, rf_grid)

gb_grid = {
    'model__n_estimators': [50,100,150,200],
    'model__learning_rate': [0.01, 0.05, 0.1, 1],
    'model__max_depth': [3, 4, 5, 6]
}
gb_pipeline = tune_model(gb_pipeline, gb_grid)

xgb_grid = {
    'model__n_estimators' : [100,200,300],
    'model__max_depth' : [3,5,7],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__subsample': [0.7, 0.8, 1],
    'model__colsample_bytree': [0.7, 0.8, 1],
    'model__eval_metric' : ['logloss']
}
xgb_pipeline = tune_model(xgb_pipeline, xgb_grid)

knn_grid = {
    'model__n_neighbors' : [11, 15, 20, 25, 30,35],
    'model__metric' : ['manhattan'],
    'model__weights' : ['distance']
}
knn_pipeline = tune_model(knn_pipeline, knn_grid)

svm_grid = {
    'model__C': [0.01, 0.1, 1, 10, 100],
    'model__gamma': [0.001, 0.01, 0.1, 0.5, 1],
    'model__kernel': ['rbf'],
    'model__probability' : [True]
}
svm_pipeline = tune_model(svm_pipeline, svm_grid)

print('Tuning Completed. Ready for next steps')

Fitting 5 folds for each of 64 candidates, totalling 320 fits
Best parameters for Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['gender', 'Partner',
                                                   'Dependents', 'PhoneService',
                                                   'MultipleLines',
                                                   'InternetService',
                                                   'OnlineSecurity',
                                                   'OnlineBackup',
                                                   'DeviceProtection',
                                                   'TechSupport', 'StreamingTV',
                                                   'StreamingMovies',
                                                   'Contract',
                          

In [40]:
#Create a class to manage the workflow and save pipelines
class Workflow:
    def __init__(self, pipeline):
        self.pipeline = pipeline

    def train(self, X_train, y_train):
        self.pipeline.fit(X_train, y_train)

    def predict_and_prob(self, X_test):
        self.y_pred = self.pipeline.predict(X_test)
        self.y_probs = self.pipeline.predict_proba(X_test)[:,1]

    def evaluate(self, y_test):
        metrics = {
            'Accuracy' : round(accuracy_score(y_test, self.y_pred),2),
            'Precision' : round(precision_score(y_test, self.y_pred),2),
            'Recall' : round(recall_score(y_test, self.y_pred),2),
            'F1 Score': round(f1_score(y_test, self.y_pred),2),
            'AUC': round(roc_auc_score(y_test, self.y_probs),2)
        }
        return metrics
        
    def save(self, filename):
        joblib.dump(self.pipeline, filename)
        print(f'Pipeline saved as {filename}')
        
    def run(self, X_train, X_test, y_train, y_test):
        self.train(X_train, y_train)
        self.predict_and_prob(X_test)
        
        return self.evaluate(y_test)
        

    @staticmethod
    def load(filename):
        pipeline = joblib.load(filename)
        print(f'Pipeline loaded from {filename}')
        return pipeline
        

In [41]:
#Train and evaluate all pipelines pipelines
train_lr = Workflow(lr_pipeline)
lr_metrics = train_lr.run(X_train, X_test, y_train, y_test)

train_dt = Workflow(dt_pipeline)
dt_metrics = train_dt.run(X_train, X_test, y_train, y_test)

train_rf = Workflow(rf_pipeline)
rf_metrics = train_rf.run(X_train, X_test, y_train, y_test)

train_gb = Workflow(gb_pipeline)
gb_metrics = train_gb.run(X_train, X_test, y_train, y_test)

train_xgb = Workflow(xgb_pipeline)
xgb_metrics = train_xgb.run(X_train, X_test, y_train, y_test)

train_knn = Workflow(knn_pipeline)
knn_metrics = train_knn.run(X_train, X_test, y_train, y_test)

train_svm = Workflow(svm_pipeline)
svm_metrics = train_svm.run(X_train, X_test, y_train, y_test)

train_nb = Workflow(nb_pipeline)
nb_metrics = train_nb.run(X_train, X_test, y_train, y_test)

In [42]:
#Ensemble models
#Voting Model
voting = VotingClassifier(
    estimators = [
        ('lr', lr_pipeline),
        ('rf', rf_pipeline),
        ('gb', gb_pipeline),
        ('xgb', xgb_pipeline)
    ],
    voting = 'soft'
)
voting_model = Workflow(voting)
voting_metrics = voting_model.run(X_train, X_test, y_train, y_test)

#stacking model
stacking = StackingClassifier(
    estimators = [
        ('lr', lr_pipeline),
        ('rf', rf_pipeline),
        ('gb', gb_pipeline),
        ('xgb', xgb_pipeline)
    ],
    final_estimator = LogisticRegression(),
    cv = 5
)
stack_model = Workflow(stacking)
stack_metrics = voting_model.run(X_train, X_test, y_train, y_test)

#Put all pipelines in a dataframe for comparison
results = []

results.append({'Model': 'Logistic Regression', **lr_metrics})
results.append({'Model': 'Decision Tree', **dt_metrics})
results.append({'Model': 'Random Forest', **rf_metrics})
results.append({'Model': 'Gradient Boosting', **gb_metrics})
results.append({'Model': 'XGBoost', **xgb_metrics})
results.append({'Model': 'kNN', **knn_metrics})
results.append({'Model': 'SVM', **svm_metrics})
results.append({'Model': 'Naive Bayes', **nb_metrics})
results.append({'Model': 'Voting Ensemble', **voting_metrics})
results.append({'Model': 'Stacking Ensemble', **stack_metrics})

compare_df = pd.DataFrame(results)
compare_df

,Model,Accuracy,Precision,Recall,F1 Score,AUC
0,Logistic Regression,0.73,0.50,0.79,0.61,0.83
1,Decision Tree,0.73,0.49,0.79,0.61,0.81
2,Random Forest,0.76,0.54,0.76,0.63,0.83
3,Gradient Boosting,0.77,0.54,0.74,0.62,0.83
4,XGBoost,0.77,0.54,0.76,0.63,0.83
5,kNN,0.76,0.53,0.70,0.60,0.80
6,SVM,0.75,0.52,0.76,0.62,0.80
7,Naive Bayes,0.72,0.48,0.78,0.59,0.82
8,Voting Ensemble,0.76,0.54,0.76,0.63,0.83
9,Stacking Ensemble,0.76,0.54,0.76,0.63,0.83


In [43]:
#save the final best pipeline for project testing and deployment
final_pipeline = Workflow(rf_pipeline)
final_pipeline.save('customer_churn_pipeline.pkl')

Pipeline saved as customer_churn_pipeline.pkl


In [44]:
#Testing the pipeline with a random saple from the dataset
#loaad the pipeline
loaded_pipeline = Workflow.load('customer_churn_pipeline.pkl')

#take sample
sample = df.drop(columns = 'Churn').iloc[[100]]

#prediction and probability
prediction = loaded_pipeline.predict(sample)
probability = loaded_pipeline.predict_proba(sample)

print(prediction)
print(probability)

print('Actual :', df.iloc[100]['Churn']) 

Pipeline loaded from customer_churn_pipeline.pkl
[0]
[[0.50058416 0.49941584]]
Actual : No


Test passed successfully!!! Pipeline ready for deployment😍